# Inference 101: Make a Prediction

Simple example: Load one customer record and get a prediction from the model.

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
import sys
from datetime import datetime

# For AI Inference Service
import httpx
from open_inference.openapi.client import OpenInferenceClient, InferenceRequest

# CRITICAL: Add paths for imports
# Add project root for shared_utils
sys.path.insert(0, os.path.abspath('..'))
# Add module1 for helpers
sys.path.insert(0, os.path.abspath('../module1'))

print("✅ Libraries imported successfully")
print(f"   Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Working directory: {os.getcwd()}")

## Step 0: Find Your Model Endpoint

To call a model you need two things: its **endpoint URL** and its **access key**.
The cell below looks them up for the model you deployed in `04_deploy.py`.
You can also find both in the UI: **Models** tab → **banking_campaign_predictor** → **Overview** → *"Instructions for importing requests"*.

In [ ]:
# Look up the endpoint URL and access key of your deployed model
import cmlapi

MODEL_NAME = "banking_campaign_predictor"
client = cmlapi.default_client()
project_id = os.environ["CDSW_PROJECT_ID"]

model = client.list_models(project_id, search_filter=json.dumps({"name": MODEL_NAME})).models[0]

print(f"Endpoint URL: https://modelservice.{os.environ['CDSW_DOMAIN']}/model")
print(f"Access key:   {model.access_key}")

**Your turn:** copy the two values printed above and paste them between the quotes below.

In [ ]:
MODEL_ENDPOINT = "PASTE_ENDPOINT_URL_HERE"
ACCESS_KEY = "PASTE_ACCESS_KEY_HERE"

if "PASTE" in MODEL_ENDPOINT or "PASTE" in ACCESS_KEY:
    raise ValueError("Paste the endpoint URL and access key from the cell above before continuing.")

print(f"✓ Using model endpoint: {MODEL_ENDPOINT}")

## Step 1: Load Single Row of Preprocessed Data

In [ ]:
# Load and preprocess data following the same process as test_endpoint.py
from helpers import PreprocessingPipeline, FeatureEngineer


# Load raw inference data
df_raw = pd.read_csv("inference_data/raw_inference_data.csv", sep=";")
print(f"Loaded raw data: {df_raw.shape}")

# Load full training data for preprocessing fit
df_train = pd.read_csv("data/bank-additional/bank-additional-full.csv", sep=";")
print(f"Loaded training data: {df_train.shape}")

# Apply feature engineering
fe = FeatureEngineer().fit(df_train)  # learn normalization from training data
df_train_eng = fe.transform(df_train)
df_engineered = fe.transform(df_raw)

# Define features
numeric_features = [
    'age', 'duration', 'campaign', 'pdays', 'previous',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed',
    'engagement_score'
]

categorical_features = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome',
    'age_group', 'emp_var_category', 'duration_category'
]

# Create and fit preprocessor on full training data
preprocessor = PreprocessingPipeline(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    include_engagement=True
)

X_train_full = df_train_eng[numeric_features + categorical_features].copy()
preprocessor.fit(X_train_full)

# Transform inference data
X_engineered = df_engineered[numeric_features + categorical_features].copy()
X_processed = pd.DataFrame(
    preprocessor.transform(X_engineered),
    columns=preprocessor.get_feature_names()
)



In [ ]:
# Pick a row number :

selected_row = 3

single_row = X_processed.iloc[[selected_row]]

print(f"Loaded and preprocessed single row")
print(f"Shape: {single_row.shape}")
print(f"Columns: {single_row.shape[1]}")

## Step 2: Format for API

In [ ]:
# Create the API payload with proper handling of NaN/inf values
row_dict = single_row.iloc[0].to_dict()

# Convert NaN and inf values for JSON serialization
for key, val in row_dict.items():
    if pd.isna(val):
        row_dict[key] = 0
    elif np.isinf(val):
        row_dict[key] = 1e10 if val > 0 else -1e10
    else:
        row_dict[key] = float(val)

payload = {
    "accessKey": ACCESS_KEY,
    "request": {
        "dataframe_split": {
            "columns": list(row_dict.keys()),
            "data": [list(row_dict.values())]
        }
    }
}

payload_json = json.dumps(payload)

print(f"Payload created")
print(f"Size: {len(payload_json)} bytes")

## Step 3: Send to Model Service

In [ ]:
# Send request to model
response = requests.post(
    MODEL_ENDPOINT,
    data=payload_json,
    headers={'Content-Type': 'application/json'},
    timeout=30
)

print(f"Response status: {response.status_code}")

## Step 4: Get the Prediction

In [ ]:
# Parse response and handle errors
print(f"Response status: {response.status_code}\n")

if response.status_code != 200:
    print("❌ Error in response!")
    print(f"Status: {response.status_code}")
    print(f"Response text: {response.text}")
else:
    try:
        result = response.json()
        prediction = result['response']['prediction'][0]

        print(f"\n" + "="*50)
        print(f"PREDICTION: {prediction}")
        print(f"="*50)

        if prediction == 0:
            print("\nResult: Customer will NOT subscribe to term deposit")
        else:
            print("\nResult: Customer WILL subscribe to term deposit")

        print(f"\nFull response:")
        print(result)
        
    except (KeyError, IndexError, ValueError) as e:
        print(f"❌ Error parsing response: {e}")
        print(f"\nResponse received:")
        print(response.json())